# ガウス過程とExpected Improvementによるベイズ最適化

既存の研究用コードから、単一目的の逐次ベイズ最適化に必要な部分だけを抽出した公開用デモです。実データ・研究結果は含みません。合成関数を有限候補集合上で最大化し、同じ初期観測点・同じ観測回数でランダム探索と比較します。

処理は「観測 → GP更新 → 予測平均と不確実性 → EI最大の候補を観測」の繰り返しです。Pythonの文法解説や初歩的な練習は省いています。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from bayesian_optimization import fit_precomputed_gp, expected_improvement, run_search

X = np.linspace(0, 1, 201).reshape(-1, 1)
# 後ろ向き評価用の合成目的関数。選択ロジックには観測済みの値のみを渡す。
y = np.sin(9 * X[:, 0]) + 0.3 * np.cos(23 * X[:, 0]) + 0.5 * X[:, 0]
length_scale = 0.08
kernel = np.exp(-0.5 * cdist(X, X, metric="sqeuclidean") / length_scale**2)


## 1. ガウス過程による予測とEI

観測値の標準化には観測集合の平均・標準偏差のみを使います。Cholesky分解で線形方程式を解き、逆行列の明示計算を避けます。EIは最大化問題に対する期待改善量です。カーネルとパラメータは固定し、ここでは調整の優劣を検討しません。


In [ ]:
initial = np.array([10, 80, 150])
candidates = np.setdiff1d(np.arange(len(y)), initial)
model = fit_precomputed_gp(kernel, y, initial)
mean, std = model.predict(candidates)
ei = expected_improvement(mean, std, y[initial].max())
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(X[:, 0], y, "--", alpha=0.4, label="Synthetic objective (evaluation only)")
axes[0].plot(X[candidates, 0], mean, label="GP mean")
axes[0].fill_between(X[candidates, 0], mean-1.96*std, mean+1.96*std, alpha=0.2)
axes[0].scatter(X[initial, 0], y[initial], color="black", label="Observed")
axes[0].legend()
axes[1].plot(X[candidates, 0], ei, label="Expected improvement")
axes[1].set_xlabel("x")
axes[1].legend()
plt.tight_layout()
plt.show()


## 2. 逐次探索とランダム探索の比較

各seedで共通の初期点を使い、20回の追加観測を行います。評価指標はbest-so-farとsimple regretです。simple regretは既知の合成関数上でのみ計算する評価値で、候補選択には使用しません。


In [ ]:
records = []
for seed in range(10):
    initial = np.random.default_rng(seed).choice(len(y), size=3, replace=False)
    for method in ("Random", "BO"):
        selected, best = run_search(kernel, y, initial, n_steps=20, method=method, seed=seed)
        assert len(np.unique(selected)) == len(selected)
        assert np.all(np.diff(best) >= -1e-12)
        for step, value in enumerate(best):
            records.append(dict(seed=seed, method=method, observations=3+step,
                                best=value, simple_regret=y.max()-value))
results = pd.DataFrame(records)
fig, ax = plt.subplots(figsize=(8, 4))
for method, rows in results.groupby("method"):
    summary = rows.groupby("observations")["simple_regret"].agg(["mean", "std"])
    ax.plot(summary.index, summary["mean"], label=method)
    ax.fill_between(summary.index, np.maximum(0, summary["mean"]-summary["std"]),
                    summary["mean"]+summary["std"], alpha=0.15)
ax.set(xlabel="Total observations", ylabel="Simple regret",
       title="Mean ± standard deviation across 10 seeds")
ax.legend()
plt.tight_layout()
plt.show()
results.groupby(["method", "observations"])["simple_regret"].mean().unstack(0).tail()


## 制限

- 合成関数上のデモであり、創薬や一般のブラックボックス問題での優位性を示すものではありません。
- カーネル・ノイズ分散・候補格子は固定しています。観測ノイズのある実験や連続空間の獲得関数最適化は扱いません。
- 研究の発展的な内容や実データは公開対象外です。
